# 2nd scenario — SB vs BB (single-raised pot), headless GPU

Solves the **SB-vs-BB** blind-vs-blind SRP across the 17-board library (the same boards, different ranges: SB opens/is OOP, BB defends/is IP). SB ranges are wider, so this runs in **3 parts** (~4–5 h each) to stay under Kaggle's ~9 h commit limit.

**Per commit:** set `PART` to `'A'`, `'B'`, then `'C'`, and Save & Run All each time (GPU T4 x2 + Internet On). Download `records_sb_<PART>.json` from each and send all three back — they merge into a signed `sb_vs_bb` pack locally.

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On) -- pulls the SB-vs-BB scenario + 17 boards.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.presets import SCENARIOS, BOARDS
print('scenario:', SCENARIOS['sb_vs_bb_srp']['label'], '| boards:', len(BOARDS))

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

## Solve one part (~4–5 h). Set PART to 'A', then 'B', then 'C' across three commits.

In [ ]:
PART = 'A'   # <-- 'A' (boards 0-5), then 'B' (6-11), then 'C' (12-16)
ROOTS = {'A': '0,1,2,3,4,5', 'B': '6,7,8,9,10,11', 'C': '12,13,14,15,16'}[PART]
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', '-m', 'pokertrainer.content_yield',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--scenario', 'sb_vs_bb_srp', '--roots', ROOTS,
     '--out', f'/kaggle/working/cy_sb_{PART}'],
    cwd='/kaggle/working/poker', env=env)

In [ ]:
# Expose records_sb_<PART>.json for download (fails loudly if nothing completed).
import json, shutil, os
try:
    rep = json.load(open(f'/kaggle/working/cy_sb_{PART}/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit(f'No cy_sb_{PART} output -- check the cell above and cy_sb_{PART}/boards/*.ERROR.txt')
done = rep.get('boards_completed') or []
print('PART', PART, '| boards completed:', done, '| missing:', rep.get('boards_missing'))
if not done:
    raise SystemExit(f'No boards completed in PART {PART} -- check cy_sb_{PART}/boards/*.ERROR.txt')
shutil.copy(f'/kaggle/working/cy_sb_{PART}/records.json', f'/kaggle/working/records_sb_{PART}.json')
print('DOWNLOAD -> records_sb_%s.json (%d KB, %d records)'
      % (PART, os.path.getsize(f'/kaggle/working/records_sb_{PART}.json')//1024,
         len(json.load(open(f'/kaggle/working/cy_sb_{PART}/records.json')))))